In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df = pd.read_csv("cleaned_statsfinal.csv")

print("Dataset loaded successfully!")
print(df.head())
print("\nShape:", df.shape)

Dataset loaded successfully!
   Unnamed: 0        Date  Q-P1  Q-P2  Q-P3  Q-P4      S-P1      S-P2  \
0           0  2010-06-13  5422  3725   576   907  17187.74  23616.50   
1           1  2010-06-14  7047   779  3578  1574  22338.99   4938.86   
2           2  2010-06-15  1572  2082   595  1145   4983.24  13199.88   
3           3  2010-06-16  5657  2399  3140  1672  17932.69  15209.66   
4           4  2010-06-17  3668  3207  2184   708  11627.56  20332.38   

       S-P3      S-P4  Total_Sales  Total_Quantity  
0   3121.92   6466.91     50393.07           10630  
1  19392.76  11222.62     57893.23           12978  
2   3224.90   8163.85     29571.87            5394  
3  17018.80  11921.36     62082.51           12868  
4  11837.28   5048.04     48845.26            9767  

Shape: (4600, 12)


In [3]:
target = "Total_Sales"

X = df.drop(columns=[target])
y = df[target]

# Convert categorical features to numeric
X = pd.get_dummies(X, drop_first=True)

# Train a baseline model to examine feature importance
model = LinearRegression()
model.fit(X, y)

importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

importance["Absolute_Importance"] = importance["Coefficient"].abs()
importance = importance.sort_values(
    by="Absolute_Importance",
    ascending=False
)

print(importance.head(10))

              Feature   Coefficient  Absolute_Importance
8                S-P4  8.915112e-01         8.915112e-01
6                S-P2  8.759233e-01         8.759233e-01
7                S-P3  8.513712e-01         8.513712e-01
5                S-P1  7.234378e-01         7.234378e-01
9      Total_Quantity  6.484883e-01         6.484883e-01
1                Q-P1  2.282138e-01         2.282138e-01
3                Q-P3  1.570796e-01         1.570796e-01
2                Q-P2  1.381582e-01         1.381582e-01
4                Q-P4  1.250366e-01         1.250366e-01
3924  Date_2021-04-10  2.057578e-07         2.057578e-07


In [4]:
most_important_feature = importance.iloc[0]["Feature"]

print("Most important feature:")
print(most_important_feature)

Most important feature:
S-P4


In [5]:
comparison = pd.read_csv("model_comparison.csv")

print(comparison)

               Model  Train R2  Test R2    Train RMSE     Test RMSE  \
0  Linear Regression       1.0      1.0  5.002132e-09  8.269709e-08   
1   Ridge Regression       1.0      1.0  8.601035e-08  1.653975e-07   
2   Lasso Regression       1.0      1.0  4.933156e-04  4.823311e-04   

         R2 Gap  
0  0.000000e+00  
1  0.000000e+00  
2  1.110223e-16  


In [6]:
best_model_name = comparison.loc[
    comparison["Test R2"].idxmax(), "Model"
]

print("Best model:", best_model_name)

if best_model_name == "Linear Regression":
    best_model = LinearRegression()
elif best_model_name == "Ridge Regression":
    best_model = Ridge(alpha=1.0)
else:
    best_model = Lasso(alpha=1.0, max_iter=10000)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

best_model.fit(X_train, y_train)

before_pred = best_model.predict(X_test)

before_r2 = r2_score(y_test, before_pred)
before_rmse = np.sqrt(mean_squared_error(y_test, before_pred))

print("R² BEFORE:", before_r2)
print("RMSE BEFORE:", before_rmse)

Best model: Linear Regression
R² BEFORE: 1.0
RMSE BEFORE: 8.269709464013443e-08


In [7]:
X_reduced = X.drop(columns=["S-P4"])

print("Removed feature: S-P4")
print("Features before:", X.shape[1])
print("Features after:", X_reduced.shape[1])

Removed feature: S-P4
Features before: 4583
Features after: 4582


In [8]:
X_train_reduced, X_test_reduced, y_train_reduced, y_test_reduced = train_test_split(
    X_reduced, y, test_size=0.2, random_state=42
)

adapted_model = best_model.__class__(**(
    {"alpha": 1.0, "max_iter": 10000}
    if best_model_name == "Lasso Regression"
    else {"alpha": 1.0}
    if best_model_name == "Ridge Regression"
    else {}
))

adapted_model.fit(X_train_reduced, y_train_reduced)

after_pred = adapted_model.predict(X_test_reduced)

after_r2 = r2_score(y_test_reduced, after_pred)
after_rmse = np.sqrt(mean_squared_error(y_test_reduced, after_pred))

print("R² AFTER:", after_r2)
print("RMSE AFTER:", after_rmse)

R² AFTER: 1.0
RMSE AFTER: 1.8928344341526092e-06


In [9]:
performance_comparison = pd.DataFrame({
    "Model": [best_model_name, best_model_name],
    "Feature_Set": ["All Features", "Without S-P4"],
    "R2_Score": [before_r2, after_r2],
    "RMSE": [before_rmse, after_rmse]
})

print(performance_comparison)

               Model   Feature_Set  R2_Score          RMSE
0  Linear Regression  All Features       1.0  8.269709e-08
1  Linear Regression  Without S-P4       1.0  1.892834e-06


In [10]:
r2_change = after_r2 - before_r2
rmse_change = after_rmse - before_rmse

print("Change in R²:", r2_change)
print("Change in RMSE:", rmse_change)

if r2_change < 0:
    print("\nR² decreased after removing S-P4.")
    print("The removed feature was important for prediction.")
else:
    print("\nR² did not decrease after removing S-P4.")
    print("The model adapted well without the feature.")

if rmse_change > 0:
    print("RMSE increased, meaning prediction error increased.")
else:
    print("RMSE did not increase.")

Change in R²: 0.0
Change in RMSE: 1.8101373395124747e-06

R² did not decrease after removing S-P4.
The model adapted well without the feature.
RMSE increased, meaning prediction error increased.


In [11]:
performance_comparison.to_csv(
    "day14_performance_comparison.csv",
    index=False
)

print("Performance comparison saved successfully!")

Performance comparison saved successfully!


# Day 14 — Sprint 2 Review

## Real-World Feature Removal Challenge

The most important feature identified from the model was `S-P4`.

To simulate a real-world situation where an important feature becomes unavailable, `S-P4` was removed from the dataset.

The best-performing model from Day 13 was then retrained using the remaining features.

## Performance Comparison

The model performance was compared before and after removing `S-P4` using:

- R² Score
- RMSE

The results are stored in `day14_performance_comparison.csv`.

## System Adaptation

Removing an important feature can reduce predictive performance because the model loses useful information.

The retrained model adapts by learning from the remaining available features. The change in R² and RMSE shows how dependent the system was on the removed feature.

This experiment demonstrates the importance of building Machine Learning systems that can handle changing data and unexpected feature loss.

## Sprint 2 Reflection

### What I Learned

During Sprint 2, I learned how a Machine Learning workflow can be affected when important data becomes unavailable. I practiced regression modeling, regularization, model comparison, and feature removal.

### Challenge I Faced

One of the main challenges was understanding how removing an important feature affects model performance and how to compare the model before and after the change.

### How the Model Adapted

After removing `S-P4`, the model was retrained using the remaining features. Comparing the R² Score and RMSE helped me understand whether the model was able to maintain its predictive performance.

### Key Takeaway

Real-world Machine Learning systems need to be flexible because datasets, features, and business requirements can change over time.